# K-Means sobre Mall Customers

Proyecto de clustering listo para **Google Colab**.

**Pregunta:** ¿podemos segmentar clientes de un mall según ingresos y gasto?

**Método:** K-Means.

**Stack:** pandas, seaborn, scikit-learn

Dataset clásico Mall Customers.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/Mounika-Kajjam/Mall-Customers/master/Mall_Customers.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.describe().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Annual Income (k$)"], kde=True, ax=axes[0])
axes[0].set_title("Ingresos anuales")
sns.histplot(df["Spending Score (1-100)"], kde=True, ax=axes[1])
axes[1].set_title("Spending Score")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)", hue="Genre")
plt.title("Ingresos vs Spending Score")
plt.show()

### Hallazgos
- Se ven grupos naturales en el scatter.
- Ideal para K-Means.

## 3. Preprocesamiento

In [ ]:
X = df[["Annual Income (k$)", "Spending Score (1-100)"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled.shape)

## 4. Elegir k (Elbow + Silhouette)

In [ ]:
inertias = []
silhouettes = []
K = range(2, 10)

for k in K:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K, inertias, marker="o")
axes[0].set_title("Elbow")
axes[1].plot(K, silhouettes, marker="o", color="green")
axes[1].set_title("Silhouette")
plt.tight_layout()
plt.show()

## 5. Clustering final

In [ ]:
k_final = 5
km = KMeans(n_clusters=k_final, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = km.fit_predict(X_scaled)
print(df["cluster"].value_counts())

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)", hue="cluster", palette="tab10")
plt.title("Clusters de clientes")
plt.show()

## 6. Perfil de segmentos

In [ ]:
perfil = df.groupby("cluster")[["Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
perfil["n"] = df["cluster"].value_counts()
perfil

## 7. Conclusiones

- Segmentos típicos: High Income High Spend, High Income Low Spend, etc.
- k=5 suele funcionar bien en este dataset.
- Dataset clásico para clustering.

**Siguiente paso:** nombrar los segmentos según negocio.